# 🦜 Clone giọng nói tiếng Việt — VieNeu-TTS v3 Turbo

Chạy trên Google Colab, **không cần cài gì trên máy**.

**Cách dùng:** bật GPU trước (`Runtime` → `Change runtime type` → `T4 GPU`),
rồi chạy lần lượt từng ô từ trên xuống.

GPU không bắt buộc nhưng nhanh hơn nhiều. CPU của Colab cũng chạy được.


## 1. Cài đặt

Mất khoảng 2–3 phút.


In [ ]:
!pip install -q vieneu
print('✅ Cài xong')


## 2. Nạp model

Lần đầu tải model từ Hugging Face, mất vài phút. Chạy lại lần sau thì nhanh.


In [ ]:
from vieneu import Vieneu
import torch

co_gpu = torch.cuda.is_available() if hasattr(torch, 'cuda') else False
print('GPU:', torch.cuda.get_device_name(0) if co_gpu else 'không có — sẽ chạy CPU')

tts = Vieneu()   # v3 Turbo, 48 kHz, tự chọn GPU hay CPU
print('✅ Model sẵn sàng')


## 3. Nghe thử 25 giọng dựng sẵn

Chạy ô này trước để biết model đọc tự nhiên tới đâu — chưa cần giọng mẫu.


In [ ]:
from IPython.display import Audio, display

for nhan, _ in tts.list_preset_voices():
    print(' -', nhan)


In [ ]:
# Đổi tên giọng và câu thử tuỳ ý
GIONG_THU = 'Minh Quân'
CAU_THU = 'Xin chào, đây là thử giọng đọc tiếng Việt. Nghe có tự nhiên không?'

audio = tts.infer(CAU_THU, voice=GIONG_THU)
display(Audio(audio, rate=48000))


## 4. Upload giọng mẫu của bạn

Chạy ô dưới rồi chọn file. Yêu cầu:

- **3–8 giây** là đủ (dài hơn sẽ bị tự cắt)
- Một người nói, đọc đều và rõ
- Không nhạc nền (có ồn nhẹ cũng được, model tự khử)
- Định dạng `.wav` hoặc `.mp3` đều được

⚠️ **Phong cách đọc đi theo clip mẫu.** Muốn giọng đọc truyện thì clip mẫu
phải là đọc truyện, muốn giọng đọc tin thì clip mẫu là đọc tin.


In [ ]:
from google.colab import files

print('Chọn file giọng mẫu...')
da_upload = files.upload()
GIONG_MAU = list(da_upload.keys())[0]
print(f'✅ Đã nhận: {GIONG_MAU}')

print('\nNghe lại clip mẫu:')
display(Audio(GIONG_MAU))


## 5. Clone giọng

Đăng ký giọng một lần rồi dùng lại theo tên.


In [ ]:
tts.add_voice('giong-cua-toi', GIONG_MAU, denoise=True)
print('✅ Đã clone')

audio = tts.infer('Đây là giọng của tôi được nhân bản.', voice='giong-cua-toi')
display(Audio(audio, rate=48000))


## 6. Đọc văn bản dài

Sửa `VAN_BAN` bên dưới. **Tách đoạn bằng dòng trống** để có khoảng lặng đúng chỗ.

Nhớ chuẩn hoá số và viết tắt trước: `2025` → `hai nghìn không trăm hai mươi lăm`,
`TP.HCM` → `Thành phố Hồ Chí Minh`.


In [ ]:
VAN_BAN = """
Chào các bạn, hôm nay mình sẽ chia sẻ một chủ đề khá thú vị.

Giọng đọc này được nhân bản từ một đoạn ghi âm rất ngắn, chỉ vài giây thôi.
Nghe khá tự nhiên phải không?

Nếu thấy hữu ích, các bạn nhớ chia sẻ cho mọi người nhé.
"""

# Chỉnh độ tự nhiên:
#   temperature thấp (0.6-0.7) -> đều và bám giọng mẫu hơn
#   temperature cao  (0.8-0.9) -> biến hoá hơn nhưng kém ổn định
TEMPERATURE = 0.7
NGHI_GIAY = 0.4          # khoảng lặng giữa các đoạn

import re
import numpy as np

doan_list = [d.strip() for d in re.split(r'\n\s*\n', VAN_BAN) if d.strip()]
print(f'{len(doan_list)} đoạn\n')

lang = np.zeros(int(48000 * NGHI_GIAY), dtype=np.float32)
phan = []
for i, doan in enumerate(doan_list, 1):
    print(f'  [{i}/{len(doan_list)}] {doan[:60]}...')
    phan.append(np.asarray(
        tts.infer(doan, voice='giong-cua-toi',
                  temperature=TEMPERATURE, repetition_penalty=1.2),
        dtype=np.float32))
    if i < len(doan_list):
        phan.append(lang)

ket_qua = np.concatenate(phan)
tts.save(ket_qua, 'ket-qua.wav')

print(f'\n✅ Xong — {len(ket_qua)/48000:.1f} giây')
display(Audio(ket_qua, rate=48000))


## 7. Tải file về


In [ ]:
files.download('ket-qua.wav')


---

## Chưa đủ giống?

Clone từ clip vài giây có giới hạn vật lý — vài giây audio không chứa đủ
thông tin để tái tạo trọn vẹn một giọng. Muốn sát hơn thì phải **fine-tune LoRA**:
cần **10–30 phút audio sạch** (100–300 câu khác nhau) và GPU ~6GB — T4 của Colab
là thừa.

Xem `finetune.sh` và `reference.md` trong skill.

## Lưu ý

Chỉ clone giọng khi **có sự đồng ý của chủ giọng**.

Colab sẽ ngắt phiên sau một thời gian — **tải file về trước khi rời đi**.
